<a href="https://colab.research.google.com/github/samritha07/Deep-Learning/blob/main/DL_EXP_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers accelerate torch gradio

In [2]:
ROLL_NOS = "24BAD103"   # <-- replace with your roll number(s)
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # small pre-trained instruction-tuned Transformer LLM
print("Roll Nos:", ROLL_NOS)
print("Model   :", MODEL_NAME)

Roll Nos: 24BAD103
Model   : Qwen/Qwen2.5-0.5B-Instruct


## PART A: Loading and Configuring a Pre-trained LLM

### A1. Select a pre-trained LLM and load the model and its tokenizer

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Roll Nos:", ROLL_NOS)
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
model.eval()
print(f"Loaded {MODEL_NAME} on {device} | parameters: {model.num_parameters()/1e6:.0f}M")

Roll Nos: 24BAD103


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-0.5B-Instruct on cpu | parameters: 494M


### A2. Prepare and tokenize sample user prompts

In [4]:
print("Roll Nos:", ROLL_NOS)
sample_prompts = [
    "What is machine learning?",
    "Explain the Transformer architecture in two sentences.",
    "Write a Python function to reverse a string.",
]
for p in sample_prompts:
    messages = [{"role": "user", "content": p}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt")
    ids = enc["input_ids"][0]
    print("Prompt        :", p)
    print("No. of tokens :", len(ids))
    print("First 10 ids  :", ids[:10].tolist())
    print("First 10 toks :", tokenizer.convert_ids_to_tokens(ids[:10]))
    print("-" * 60)

Roll Nos: 24BAD103
Prompt        : What is machine learning?
No. of tokens : 34
First 10 ids  : [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553]
First 10 toks : ['<|im_start|>', 'system', 'Ċ', 'You', 'Ġare', 'ĠQ', 'wen', ',', 'Ġcreated', 'Ġby']
------------------------------------------------------------
Prompt        : Explain the Transformer architecture in two sentences.
No. of tokens : 38
First 10 ids  : [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553]
First 10 toks : ['<|im_start|>', 'system', 'Ċ', 'You', 'Ġare', 'ĠQ', 'wen', ',', 'Ġcreated', 'Ġby']
------------------------------------------------------------
Prompt        : Write a Python function to reverse a string.
No. of tokens : 38
First 10 ids  : [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553]
First 10 toks : ['<|im_start|>', 'system', 'Ċ', 'You', 'Ġare', 'ĠQ', 'wen', ',', 'Ġcreated', 'Ġby']
------------------------------------------------------------


### A3. Generate responses with temperature and maximum output length

In [5]:
print("Roll Nos:", ROLL_NOS)

def generate_response(prompt, temperature=0.7, max_new_tokens=150, top_p=0.9):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    gen_kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=top_p)
    else:
        gen_kwargs.update(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(generate_response("What is machine learning?", temperature=0.7, max_new_tokens=100))

Roll Nos: 24BAD103
Machine learning is a subset of artificial intelligence that enables computers to learn from data and improve their performance over time without being explicitly programmed. In simpler terms, it's about training algorithms on datasets so that they can make predictions or decisions based on patterns found in the data.

There are different types of machine learning techniques, including supervised learning, unsupervised learning, and reinforcement learning. Supervised learning involves using labeled data to train an algorithm to predict outcomes based on input features. Unsupervised learning uses unl


### A4. Display and examine the generated responses (effect of temperature and length)

In [6]:
print("Roll Nos:", ROLL_NOS)
prompt = "Give me a creative name for a coffee shop and a one-line slogan."

for temp in [0.0, 0.5, 1.0, 1.5]:
    print(f"\n=== temperature = {temp} ===")
    print(generate_response(prompt, temperature=temp, max_new_tokens=60))

print("\n\n##### Effect of max_new_tokens #####")
for n in [20, 80]:
    r = generate_response("Explain what an LLM is.", temperature=0.5, max_new_tokens=n)
    print(f"\n--- max_new_tokens = {n} (words: {len(r.split())}) ---\n{r}")

Roll Nos: 24BAD103

=== temperature = 0.0 ===
Creative Name: "Café Bliss"

One-Line Slogan: "Savor the Moment, Enjoy the Experience."

=== temperature = 0.5 ===
Creative Name: "Caffeine Haven"

One-Line Slogan: "Savor the Moment with Our Freshly Brewed Coffee"

=== temperature = 1.0 ===
Name: "Aroma Lane"

Slogan: "Smell the Way Home - Aroma Coffee Experience"

=== temperature = 1.5 ===
A creative name could be "Grind and Enjoy: The Coffee Haven". A one-line slogan that reflects this brand could be:

"Unleash the best in coffee every morning. Here at The Grindery, we're not just about serving coffee; we're here to make your experience a


##### Effect of max_new_tokens #####

--- max_new_tokens = 20 (words: 17) ---
An LLM (Large Language Model) refers to a type of artificial intelligence model that can generate human

--- max_new_tokens = 80 (words: 67) ---
An LLM (Large Language Model) refers to a machine learning model that can generate human-like text based on inputs provided by the

## PART B: Implementing Chatbot Functionality

### B1. Create a function to accept user queries and generate responses

In [7]:
print("Roll Nos:", ROLL_NOS)

def chatbot_reply(user_query, temperature=0.7, max_new_tokens=150):
    """Single-turn chatbot: accepts a query and returns the LLM's answer."""
    system_prompt = "You are a helpful, concise assistant."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=0.9 if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(chatbot_reply("Who developed the Python language?"))

Roll Nos: 24BAD103
The Python programming language was developed by Guido van Rossum and first published in 1989 as an extension to the ABC family of languages. He coined the term "Python" himself and is known for his contributions to object-oriented programming and creating a modern, readable syntax that is now widely used. Python has since become one of the most popular programming languages globally.


### B2. Construct suitable prompts and display the generated responses

In [8]:
print("Roll Nos:", ROLL_NOS)
queries = [
    "Explain overfitting in simple words.",
    "Give three tips to prepare for an exam.",
    "Translate 'Good morning' into French.",
]
for q in queries:
    print("User   :", q)
    print("Chatbot:", chatbot_reply(q, temperature=0.5, max_new_tokens=100))
    print("-" * 60)

Roll Nos: 24BAD103
User   : Explain overfitting in simple words.
Chatbot: Overfitting occurs when a model is too complex and fits the training data too closely, resulting in poor performance on new or unseen data. It happens because the model has learned features that are specific to the training dataset rather than general patterns in the data. This leads to overconfidence in the model's ability to generalize and can result in underperformance on new data. To avoid overfitting, it's important to use regularization techniques such as dropout, early stopping, or L1/L2 regularization.
------------------------------------------------------------
User   : Give three tips to prepare for an exam.
Chatbot: 1. Create a study schedule and stick to it.
2. Practice tests regularly to reinforce your learning.
3. Get enough sleep and eat well before the exam.
------------------------------------------------------------
User   : Translate 'Good morning' into French.
Chatbot: "Good morning" translate

### B3. Maintain conversation history

In [9]:
print("Roll Nos:", ROLL_NOS)

class ChatSession:
    def __init__(self, system_prompt="You are a helpful, concise assistant.", max_turns=6):
        self.system = {"role": "system", "content": system_prompt}
        self.history = []          # list of {"role": ..., "content": ...}
        self.max_turns = max_turns # keep only the last N exchanges to limit context length

    def ask(self, user_msg, temperature=0.7, max_new_tokens=150):
        self.history.append({"role": "user", "content": user_msg})
        trimmed = self.history[-2 * self.max_turns:]
        text = tokenizer.apply_chat_template([self.system] + trimmed, tokenize=False,
                                             add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=temperature > 0,
                temperature=temperature if temperature > 0 else None,
                top_p=0.9 if temperature > 0 else None,
                pad_token_id=tokenizer.eos_token_id)
        reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        self.history.append({"role": "assistant", "content": reply})
        return reply

    def reset(self):
        self.history = []

session = ChatSession()
for q in ["My name is Priya and I am studying AI.",
          "What is my name?",
          "What subject am I studying?"]:
    print("User   :", q)
    print("Chatbot:", session.ask(q, temperature=0.3, max_new_tokens=60))
    print("-" * 60)

Roll Nos: 24BAD103
User   : My name is Priya and I am studying AI.
Chatbot: Hello Priya! It's nice to meet you. Is there anything specific you'd like to discuss or ask about AI?
------------------------------------------------------------
User   : What is my name?
Chatbot: Your name is Priya.
------------------------------------------------------------
User   : What subject am I studying?
Chatbot: You are currently studying Artificial Intelligence (AI).
------------------------------------------------------------


### B4. Test the chatbot with different queries and observe response quality

In [10]:
print("Roll Nos:", ROLL_NOS)
import time

test_queries = {
    "Factual"   : "What is the capital of Japan?",
    "Reasoning" : "If a train travels 60 km in 1.5 hours, what is its average speed?",
    "Creative"  : "Write a two-line poem about the rain.",
    "Coding"    : "Write a Python one-liner to find the sum of a list called nums.",
    "Ambiguous" : "Tell me about Java.",
}
rows = []
for cat, q in test_queries.items():
    t0 = time.time()
    ans = chatbot_reply(q, temperature=0.3, max_new_tokens=100)
    dt = time.time() - t0
    rows.append((cat, q, ans, dt))
    print(f"[{cat}] {q}\n  -> {ans}\n  (time: {dt:.2f}s)\n")

Roll Nos: 24BAD103
[Factual] What is the capital of Japan?
  -> The capital of Japan is Tokyo.
  (time: 1.60s)

[Reasoning] If a train travels 60 km in 1.5 hours, what is its average speed?
  -> To calculate the average speed of the train, you use the formula:

\[ \text{Average Speed} = \frac{\text{Total Distance}}{\text{Total Time}} \]

Given that the train traveled 60 km in 1.5 hours, you can plug these values into the formula:

\[ \text{Average Speed} = \frac{60 \text{ km}}{1.5 \text{ hours}} \]

\[ \text{Average Speed} =
  (time: 16.61s)

[Creative] Write a two-line poem about the rain.
  -> The rain falls like silver drops,
Whispering secrets to the earth.
  (time: 3.20s)

[Coding] Write a Python one-liner to find the sum of a list called nums.
  -> nums = [1, 2, 3, 4]
sum(nums)
  (time: 3.67s)

[Ambiguous] Tell me about Java.
  -> Java is an open-source programming language developed by Sun Microsystems in the late 1990s. It was designed to be platform-independent and has since b

## PART C: Developing an Interactive Interface (Gradio)

### C1. Select Gradio for the interface and import it

In [11]:
print("Roll Nos:", ROLL_NOS)
import gradio as gr
print("Gradio version:", gr.__version__)

Roll Nos: 24BAD103
Gradio version: 6.31.0


### C2. Connect the interface to the LLM response-generation function

In [12]:
print("Roll Nos:", ROLL_NOS)

def _to_messages(history):
    """Accept both Gradio history formats (list of dicts or list of [user, bot] pairs)."""
    msgs = []
    for h in history:
        if isinstance(h, dict):
            msgs.append({"role": h["role"], "content": h["content"] if isinstance(h["content"], str) else str(h["content"])})
        else:
            u, b = h
            if u: msgs.append({"role": "user", "content": u})
            if b: msgs.append({"role": "assistant", "content": b})
    return msgs

def gradio_respond(message, history, temperature, max_new_tokens):
    msgs = [{"role": "system", "content": "You are a helpful, concise assistant."}]
    msgs += _to_messages(history)[-12:]
    msgs.append({"role": "user", "content": message})
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=int(max_new_tokens), do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=0.9 if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(gradio_respond("Hello! Who are you?", [], 0.5, 50))

Roll Nos: 24BAD103
I am an AI assistant created by Anthropic to be helpful, harmless, and honest. How can I assist you today?


### C3. Create text input, submission mechanism and chat-style layout; run the app

In [13]:
print("Roll Nos:", ROLL_NOS)

demo = gr.ChatInterface(
    fn=gradio_respond,
    additional_inputs=[
        gr.Slider(0.0, 1.5, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(20, 400, value=150, step=10, label="Max output length (tokens)"),
    ],
    title="Transformer LLM Chatbot",
    description=f"Chatbot using {MODEL_NAME} | Roll Nos: {ROLL_NOS}",
)
demo.launch(share=True, debug=False)   # opens a chat-style UI; share=True gives a public link in Colab

Roll Nos: 24BAD103
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://da4dd12e68aa0921f1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### C4. Run and test the application with different user inputs (programmatic check)

In [14]:
print("Roll Nos:", ROLL_NOS)
for q in ["Hi!", "What is deep learning?", "Give me a fun fact about space."]:
    print("User   :", q)
    print("Chatbot:", gradio_respond(q, [], 0.5, 80))
    print("-" * 60)
# Try the same inputs in the Gradio window above and take a screenshot for the record.

Roll Nos: 24BAD103
User   : Hi!
Chatbot: Hello! How can I assist you today?
------------------------------------------------------------
User   : What is deep learning?
Chatbot: Deep learning is a type of machine learning that uses neural networks with many layers to learn complex patterns and relationships in data. It involves training the network on large datasets using backpropagation to adjust weights over time based on errors made during training. Deep learning algorithms can be applied to a wide range of tasks such as image recognition, natural language processing, speech recognition, and predictive analytics.
------------------------------------------------------------
User   : Give me a fun fact about space.
Chatbot: One fun fact about space is that it's the only place we know for certain where life as we know it exists.
------------------------------------------------------------


## PART D: Application Testing and Deployment

### D1. Test response quality, latency and hallucinations

In [15]:
print("Roll Nos:", ROLL_NOS)
import time, pandas as pd

latency = []
for n in [25, 50, 100, 200]:
    t0 = time.time()
    r = chatbot_reply("Explain how neural networks learn.", temperature=0.5, max_new_tokens=n)
    dt = time.time() - t0
    ntok = len(tokenizer(r)["input_ids"])
    latency.append({"max_new_tokens": n, "tokens_generated": ntok,
                    "time_sec": round(dt, 2), "tokens_per_sec": round(ntok / dt, 1)})
print(pd.DataFrame(latency))

Roll Nos: 24BAD103
   max_new_tokens  tokens_generated  time_sec  tokens_per_sec
0              25                25      4.44             5.6
1              50                50      9.64             5.2
2             100               100     15.93             6.3
3             200               200     33.07             6.0


In [16]:
print("Roll Nos:", ROLL_NOS)
# Hallucination probe: questions about things that do not exist / cannot be known.
probes = [
    "Who won the Nobel Prize in Physics in 2089?",
    "Summarise the novel 'The Glass Clockmaker of Zorvania' by Ela Marquez.",
    "What is the phone number of the Principal of my college?",
]
for q in probes:
    print("User   :", q)
    print("Chatbot:", chatbot_reply(q, temperature=0.3, max_new_tokens=80))
    print("-" * 60)
print("Observation: a small LLM often answers confidently with invented facts (hallucination) "
      "instead of saying it does not know.")

Roll Nos: 24BAD103
User   : Who won the Nobel Prize in Physics in 2089?
Chatbot: The Nobel Prize in Physics was awarded to John Baez and David Tong for their work on "Quantum gravity."
------------------------------------------------------------
User   : Summarise the novel 'The Glass Clockmaker of Zorvania' by Ela Marquez.
Chatbot: "Zorvania" is a novel by Ela Marquez that follows the story of a young woman named Zora who moves to a small town in Zorvania after her family's farm falls into ruin. She quickly becomes friends with a local man named Sam and learns about his business as a glass clockmaker. As she spends more time with Sam, she begins to notice strange occurrences
------------------------------------------------------------
User   : What is the phone number of the Principal of my college?
Chatbot: I'm sorry, but I don't have any specific information about your college or its principal's phone number without more context. Could you please provide me with the name of your col

### D2. Organise the source code - create `app.py`

In [17]:
%%writefile app.py
# Transformer-based LLM chatbot using Gradio
# Roll Nos: <ROLL_NO_1>, <ROLL_NO_2>
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
model.eval()


def _to_messages(history):
    msgs = []
    for h in history:
        if isinstance(h, dict):
            content = h["content"] if isinstance(h["content"], str) else str(h["content"])
            msgs.append({"role": h["role"], "content": content})
        else:
            u, b = h
            if u:
                msgs.append({"role": "user", "content": u})
            if b:
                msgs.append({"role": "assistant", "content": b})
    return msgs


def respond(message, history, temperature, max_new_tokens):
    msgs = [{"role": "system", "content": "You are a helpful, concise assistant."}]
    msgs += _to_messages(history)[-12:]
    msgs.append({"role": "user", "content": message})
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=0.9 if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


demo = gr.ChatInterface(
    fn=respond,
    additional_inputs=[
        gr.Slider(0.0, 1.5, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(20, 400, value=150, step=10, label="Max output length (tokens)"),
    ],
    title="Transformer LLM Chatbot",
    description="Chatbot using Qwen2.5-0.5B-Instruct | Roll Nos: <ROLL_NO_1>, <ROLL_NO_2>",
)

if __name__ == "__main__":
    demo.launch()

Writing app.py


### D3. Create `requirements.txt` and `README.md`

In [18]:
%%writefile requirements.txt
transformers
accelerate
torch
gradio

Writing requirements.txt


In [19]:
%%writefile README.md
# Transformer-Based LLM Chatbot (Gradio)
Chatbot built with Hugging Face Transformers (Qwen2.5-0.5B-Instruct) and a Gradio chat interface.

## Run locally
```
pip install -r requirements.txt
python app.py
```

## Features
- Chat-style interface with conversation history
- Adjustable temperature and maximum output length

## Limitations
- Can hallucinate (confident but wrong answers)
- Latency on CPU; limited context length and knowledge cut-off
- Small model: weaker reasoning than large LLMs

Roll Nos: <ROLL_NO_1>, <ROLL_NO_2>

Writing README.md


In [20]:
!ls -l app.py requirements.txt README.md

-rw-r--r-- 1 root root 2189 Nov  4 17:13 app.py
-rw-r--r-- 1 root root  533 Nov  4 17:13 README.md
-rw-r--r-- 1 root root   37 Nov  4 17:13 requirements.txt
